# 07. Variational Autoencoders (VAEs)

Introduced by Kingma & Welling (2013), **Variational Autoencoders** introduce probabilistic Bayesian rigor to autoencoders by modeling latent space as continuous multivariate probability distributions.

### The Reparameterization Trick:
Backpropagation cannot flow through a stochastic sampling operation $\mathbf{z} \sim \mathcal{N}(\mu, \sigma^2)$. The reparameterization trick factors randomness out:
$$\mathbf{z} = \mu + \sigma \odot \epsilon, \quad \text{where } \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

### Loss: Evidence Lower Bound (ELBO):
$$\mathcal{L}_{\text{VAE}} = \text{Reconstruction Loss } \|\mathbf{x} - \hat{\mathbf{x}}\|^2 + D_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, \mathbf{I}))$$

In [ ]:
import torch
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, in_features=16, latent_dim=4):
        super().__init__()
        self.encoder_shared = nn.Linear(in_features, 8)
        self.fc_mu = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, in_features))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = torch.relu(self.encoder_shared(x))
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

vae = VAE()
x_in = torch.randn(2, 16)
recon, mu, logvar = vae(x_in)
print("Reconstructed Output shape:", recon.shape)
print("Latent Mean shape:         ", mu.shape)

## 🎯 Exercises & Practice
1. Explain what the Kullback-Leibler (KL) divergence penalty enforces in the latent space.

In [ ]:
# Exercise Solution
# Answer: It forces the learned latent distributions to cluster near a standard Gaussian N(0, I),
# preventing clusters from drifting apart and ensuring smooth interpolation without empty dead zones!